# SAM2 whole-plant sanity check

This is a diagnostic audit of Phase 4B. It reuses the same SAM 2.1 Hiera Tiny checkpoint and exact prompt coordinates. It does not change prompts, models, Phase 4B outputs, masks, or raw images.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start):
    for candidate in (start,*start.parents):
        if (candidate/'src').is_dir() and (candidate/'data'/'raw'/'first_sample').is_dir(): return candidate
    raise FileNotFoundError('Open from the thesis project or its notebooks folder.')
ROOT=find_project_root(Path.cwd().resolve()); sys.path.insert(0,str(ROOT)); RESULTS=ROOT/'results'/'sam2_sanity_check'


## Scope

The audit compares MPS and CPU using image_07 and image_08 for all strategies, plus image_06 as a clearer control. Device-agreement IoU is agreement between model outputs only, **not** segmentation accuracy. SAM2 is requested with `multimask_output=False`, so exactly one candidate is returned; index 0 is selected and paired with score index 0.

In [ ]:
import subprocess
for image_id in ['image_06','image_07','image_08']:
    subprocess.run([sys.executable,str(ROOT/'scripts'/'run_phase4b_sanity.py'),'--image-id',image_id],check=True)

## Raw-output and dimension audit

The inspection JSON records Python type, shape, dtype, candidate count, score shape, logits shape/range, selected-index logic, and binary values. `shape_stages.csv` verifies each stage remains H×W aligned with the original image. No local mask resizing occurs; SAM2 performs its own internal processing before returning a source-sized mask.

In [ ]:
import json,pandas as pd
inspection=json.loads((RESULTS/'raw_output_inspection.json').read_text()); display(pd.DataFrame(inspection)); display(pd.read_csv(RESULTS/'shape_stages.csv'))

## Binary save/load and visualization audit

Raw binary masks are written before plotting, as 0/255 PNGs. Diagnostic figures compare the in-memory SAM2 mask, reloaded PNG, and overlay generated from the same in-memory mask. No morphology, smoothing, thresholding, or interpolation is applied by this audit.

In [ ]:
device=pd.read_csv(RESULTS/'device_comparison.csv'); agreement=pd.read_csv(RESULTS/'mps_cpu_agreement.csv'); structure=pd.read_csv(RESULTS/'mask_structure.csv'); display(device); display(agreement); display(structure)

## Concise report

Interpret stripe patterns only after checking raw masks, PNG round trips, figures, and device agreement. A stripe seen in raw masks and the reloaded PNG is not introduced by plotting or disk serialization. A strong MPS/CPU disagreement may indicate backend-specific behaviour; close agreement on both devices suggests genuine model-output behaviour for the prompts and difficult scenes.

In [ ]:
roundtrip=[r['roundtrip_differing_pixels'] for r in inspection]; print('Model/checkpoint: SAM 2.1 Hiera Tiny / sam2.1_hiera_tiny.pt'); print('Images:',sorted(device.Image_ID.unique())); print('Strategies:',sorted(device.Prompt_Strategy.unique())); print('MPS available:', 'mps' in set(device.Device)); print('Shapes correct:', bool((device.Mask_Height==4640).all() and (device.Mask_Width==2088).all())); print('Selected index: 0 of one returned candidate for every run.'); print('Custom mask resizing: none. Binarization: SAM2 boolean mask → uint8 0/255 only when saving PNG.'); print('PNG round-trip differing pixels:',max(roundtrip)); print('Mean device agreement IoU:',round(agreement.MPS_vs_CPU_IoU.mean(),4)); print('Conclusion must follow the raw-mask/CPU-MPS evidence, not internal SAM2 scores.')